#### Imports

In [15]:
import json
from pathlib import Path
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import random_split, Dataset, DataLoader
import open_clip


#### Load data

In [16]:
IMAGE_DIR = "RSICD_images"
DATA_B_JSON = "processed_data/dataB.json"

with open(DATA_B_JSON, "r") as f:
    dataB = json.load(f)["images"]


#### Load model

In [17]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai"
)
tokenizer = open_clip.get_tokenizer("ViT-B-32")

model = model.to(device)

/home/arviv790/.local/lib/python3.10/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


#### Prep data


In [18]:
generator = torch.Generator().manual_seed(42)

train_size = int(0.8 * len(dataB))
val_size = len(dataB) - train_size

train_items, val_items = random_split(
    dataB,
    [train_size, val_size],
    generator=generator
)

In [19]:
def make_pairs(items):
    pairs = []

    for item in items:
        for sentence in item["sentences"]:
            caption = sentence["raw"].strip()

            pairs.append({
                "filename": item["filename"],
                "caption": caption,
                "label": item["label"]
            })

    return pairs

train_pairs = make_pairs(train_items)
val_pairs = make_pairs(val_items)

print(len(train_pairs), len(val_pairs))
print(train_pairs[0])

4375 1095
{'filename': 'commercial_8.jpg', 'caption': 'the business area is very large .', 'label': 'Commercial'}


In [20]:
class RSICDCaptionDataset(Dataset):
    def __init__(self, pairs, image_dir, preprocess, tokenizer):
        self.pairs = pairs
        self.image_dir = Path(image_dir)
        self.preprocess = preprocess
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        item = self.pairs[idx]

        image_path = self.image_dir / item["filename"]

        image = Image.open(image_path).convert("RGB")
        image = self.preprocess(image)

        text = self.tokenizer([item["caption"]])[0]
        label = item["label"]

        return image, text, label

In [21]:
train_dataset = RSICDCaptionDataset(train_pairs, IMAGE_DIR, preprocess, tokenizer)
val_dataset = RSICDCaptionDataset(val_pairs, IMAGE_DIR, preprocess, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [22]:
print("check sample")
image, text, label = train_dataset[0]

print(image.shape)
print(text.shape)
print(label)
print(image.dtype)
print(text.dtype)

print("check batch")
images, texts, labels = next(iter(train_loader))

print(images.shape)
print(texts.shape)
print(labels[:5])

check sample
torch.Size([3, 224, 224])
torch.Size([77])
Commercial
torch.float32
torch.int64
check batch


torch.Size([32, 3, 224, 224])
torch.Size([32, 77])
('BaseballField', 'Mountain', 'DenseResidential', 'Mountain', 'playfields')


#### Freeze layers

In [23]:
for p in model.parameters():
    p.requires_grad = False

for p in model.visual.transformer.resblocks[-1].parameters():
    p.requires_grad = True

model.visual.proj.requires_grad = True
model.text_projection.requires_grad = True
model.logit_scale.requires_grad = True

#### Optimizer

In [24]:
optimizer = torch.optim.AdamW([
    {"params": model.visual.parameters(), "lr": 5e-6},
    {"params": [model.text_projection], "lr": 5e-6},
    {"params": [model.logit_scale], "lr": 5e-6},
])

#### Loss

In [25]:
def clip_loss(image_features, text_features, logit_scale):
    image_features = F.normalize(image_features, dim=-1)
    text_features = F.normalize(text_features, dim=-1)

    logits = logit_scale.exp() * image_features @ text_features.T
    targets = torch.arange(len(image_features), device=image_features.device)

    loss_i2t = F.cross_entropy(logits, targets)
    loss_t2i = F.cross_entropy(logits.T, targets)

    return (loss_i2t + loss_t2i) / 2

#### F1 evaluation function

In [26]:
class_names = sorted({item["label"] for item in dataB if item["label"] != "Unknown"})
class_to_idx = {c: i for i, c in enumerate(class_names)}

prompts = [f"a satellite image of {name}" for name in class_names]
class_text_tokens = tokenizer(prompts).to(device)

def evaluate_zero_shot_f1(model, loader):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        text_features = model.encode_text(class_text_tokens)
        text_features = F.normalize(text_features, dim=-1)

        for images, texts, labels in loader:
            images = images.to(device)

            image_features = model.encode_image(images)
            image_features = F.normalize(image_features, dim=-1)

            logits = image_features @ text_features.T
            pred_indices = logits.argmax(dim=1)

            preds = [class_names[i] for i in pred_indices.cpu().tolist()]

            all_preds.extend(preds)
            all_labels.extend(labels)

    y_true = [class_to_idx[y] for y in all_labels]
    y_pred = [class_to_idx[p] for p in all_preds]

    accuracy = sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)

    num_classes = len(class_names)

    tp = [0] * num_classes
    fp = [0] * num_classes
    fn = [0] * num_classes

    for t, p in zip(y_true, y_pred):
        if t == p:
            tp[t] += 1
        else:
            fp[p] += 1
            fn[t] += 1

    f1_sum = 0.0

    for i in range(num_classes):
        precision = tp[i] / (tp[i] + fp[i] + 1e-8)
        recall = tp[i] / (tp[i] + fn[i] + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        f1_sum += f1

    macro_f1 = f1_sum / num_classes

    return accuracy, macro_f1

#### Training

In [27]:
#test forward pass
images, texts, label = next(iter(train_loader))
images = images.to(device)
texts = texts.to(device)

image_features = model.encode_image(images)
text_features = model.encode_text(texts)

loss = clip_loss(image_features, text_features, model.logit_scale)
print(loss.item())

print("device:", device)
print("cuda available:", torch.cuda.is_available())
print("model device:", next(model.parameters()).device)

1.9434486627578735
device: cuda
cuda available: True
model device: cuda:0


In [28]:
num_epochs = 15
best_val_f1 = -1.0

for epoch in range(num_epochs):
    model.train()
    train_loss = 0

    for images, texts, label in train_loader:
        images = images.to(device)
        texts = texts.to(device)

        image_features = model.encode_image(images)
        text_features = model.encode_text(texts)

        loss = clip_loss(image_features, text_features, model.logit_scale)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    train_loss /= len(train_dataset)

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for images, texts, label in val_loader:
            images = images.to(device)
            texts = texts.to(device)

            image_features = model.encode_image(images)
            text_features = model.encode_text(texts)

            loss = clip_loss(image_features, text_features, model.logit_scale)
            val_loss += loss.item() * images.size(0)

    val_loss /= len(val_dataset)

    val_acc, val_f1 = evaluate_zero_shot_f1(model, val_loader)

    print(
        f"Epoch {epoch+1}: "
        f"train loss={train_loss:.4f}, "
        f"val loss={val_loss:.4f}, "
        f"val acc={val_acc:.4f}, "
        f"val F1={val_f1:.4f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), "clip_trained_model.pt")
        print("Saved best model")

Epoch 1: train loss=1.5445, val loss=2.1894, val acc=0.6301, val F1=0.5693
Saved best model
Epoch 2: train loss=1.1289, val loss=2.1457, val acc=0.6804, val F1=0.6213
Saved best model
Epoch 3: train loss=0.9316, val loss=2.1386, val acc=0.7397, val F1=0.7070
Saved best model
Epoch 4: train loss=0.7676, val loss=2.1549, val acc=0.7534, val F1=0.7044
Epoch 5: train loss=0.6603, val loss=2.1735, val acc=0.7626, val F1=0.7179
Saved best model
Epoch 6: train loss=0.5485, val loss=2.1889, val acc=0.7763, val F1=0.7345
Saved best model
Epoch 7: train loss=0.4838, val loss=2.2169, val acc=0.7489, val F1=0.7169
Epoch 8: train loss=0.4218, val loss=2.2271, val acc=0.7671, val F1=0.7277
Epoch 9: train loss=0.3667, val loss=2.2532, val acc=0.7763, val F1=0.7534
Saved best model
Epoch 10: train loss=0.3298, val loss=2.2609, val acc=0.7808, val F1=0.7490
Epoch 11: train loss=0.2921, val loss=2.2775, val acc=0.7808, val F1=0.7557
Saved best model
Epoch 12: train loss=0.2892, val loss=2.2857, val acc=